# 02_17 Kaggle 3D SwinUNETR PHE-only fair baseline

This replaces the previous U-Mamba attempt. It trains a PHE-only 3D Swin Transformer U-Net baseline on Kaggle T4 using MONAI `SwinUNETR`.

Fairness lock vs `02_12` / `16b` style baselines:

- Same dataset: PHE-SICH CT only.
- Same target: binary manual PHE mask only (`background=0`, `PHE=1`).
- Same locked split: train 99, val 10, test 11.
- No ICH teacher, no pseudo ICH, no prior channel, no external labels.
- Main changed factor: 3D SwinUNETR model family instead of vanilla nnU-Net / other baselines.

Kaggle note: enable GPU T4. This notebook installs/updates MONAI and optionally clones the official MONAI repo into `/kaggle/working/MONAI` for provenance, while importing the stable pip package for training.


In [1]:
from pathlib import Path
import os
import sys
import re
import json
import time
import math
import random
import shutil
import subprocess
import gc

import numpy as np
import pandas as pd

IS_KAGGLE = Path('/kaggle/input').exists()
KAGGLE_INPUT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path.cwd().resolve()
LOCAL_ROOT = Path.cwd().resolve()

OUTPUT_ROOT = WORK_ROOT / 'outputs_02_17_kaggle_swinunetr_phe_only_fair_baseline'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
PRED_DIR = OUTPUT_ROOT / 'predictions_best_model'
METRIC_DIR = OUTPUT_ROOT / 'metrics'
CACHE_DIR = OUTPUT_ROOT / 'monai_cache'
for p in [OUTPUT_ROOT, CHECKPOINT_DIR, PRED_DIR, METRIC_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Optional manual override if auto-detect picks the wrong Kaggle dataset folder.
# Example:
# USER_PHE_ROOT = Path('/kaggle/input/phe-sich-ct-ids/PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT')
USER_PHE_ROOT = None

# Locked 02_12 split: train 99, val 10, test 11.
TRAIN_SCAN_IDS = {
    '0001', '0004', '0005', '0006', '0008', '0009', '0011', '0012', '0015', '0016',
    '0017', '0018', '0021', '0022', '0023', '0024', '0025', '0026', '0028', '0030',
    '0031', '0032', '0034', '0035', '0036', '0037', '0038', '0039', '0040', '0041',
    '0042', '0043', '0044', '0045', '0046', '0048', '0049', '0052', '0053', '0055',
    '0056', '0057', '0058', '0059', '0064', '0065', '0066', '0067', '0068', '0069',
    '0070', '0071', '0073', '0074', '0075', '0076', '0077', '0081', '0083', '0086',
    '0087', '0088', '0090', '0092', '0093', '0097', '0098', '0099', '0100', '0101',
    '0102', '0103', '0104', '0106', '0108', '0112', '0118', '0124', '0125', '0126',
    '0129', '0131', '0132', '0133', '0134', '0135', '0136', '0139', '0140', '0141',
    '0142', '0144', '0146', '0161', '0162', '0163', '0164', '0165', '0166'
}
VAL_SCAN_IDS = {'0013', '0014', '0060', '0078', '0080', '0115', '0119', '0130', '0138', '0160'}
TEST_SCAN_IDS = {'0002', '0029', '0033', '0051', '0061', '0084', '0095', '0096', '0113', '0116', '0167'}
EXPECTED_SCAN_IDS = TRAIN_SCAN_IDS | VAL_SCAN_IDS | TEST_SCAN_IDS

SEED = 20260217
MODEL_NAME = 'monai_swinunetr_3d_phe_only'
MAX_EPOCHS = 250
VAL_INTERVAL = 5
BATCH_SIZE = 1
NUM_SAMPLES_PER_CASE = 2
NUM_WORKERS = 2 if IS_KAGGLE else 0
CACHE_RATE_TRAIN = 0.15 if IS_KAGGLE else 0.0
CACHE_RATE_VAL_TEST = 0.50 if IS_KAGGLE else 0.0

# T4-safe defaults. Spatial dims must be divisible by 32 for SwinUNETR.
ROI_SIZE = (96, 96, 32)
SW_BATCH_SIZE = 4
SLIDING_OVERLAP = 0.50
FEATURE_SIZE = 24
USE_GRADIENT_CHECKPOINTING = True
AMP = True
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
GRAD_CLIP_NORM = 12.0

RUN_DEP_SETUP = True
CLONE_MONAI_REPO = True
INSTALL_MONAI = True
MONAI_PACKAGE_SPEC = 'monai>=1.5.0'
MONAI_REPO_URL = 'https://github.com/Project-MONAI/MONAI.git'
MONAI_CLONE_DIR = WORK_ROOT / 'MONAI'

RUN_TRAIN = True
RESUME_TRAINING = True
RUN_PREDICT_AND_EVALUATE = True

print('IS_KAGGLE       :', IS_KAGGLE)
print('WORK_ROOT       :', WORK_ROOT)
print('OUTPUT_ROOT     :', OUTPUT_ROOT)
print('Split sizes     :', len(TRAIN_SCAN_IDS), len(VAL_SCAN_IDS), len(TEST_SCAN_IDS))
print('ROI_SIZE        :', ROI_SIZE)
print('MAX_EPOCHS      :', MAX_EPOCHS)
print('MODEL_NAME      :', MODEL_NAME)


IS_KAGGLE       : True
WORK_ROOT       : /kaggle/working
OUTPUT_ROOT     : /kaggle/working/outputs_02_17_kaggle_swinunetr_phe_only_fair_baseline
Split sizes     : 99 10 11
ROI_SIZE        : (96, 96, 32)
MAX_EPOCHS      : 250
MODEL_NAME      : monai_swinunetr_3d_phe_only


## 0. Kaggle setup

This cell avoids the old U-Mamba dependency trap. It keeps Kaggle's existing Torch/CUDA stack, installs a current MONAI package, and optionally clones the official MONAI repository for reproducibility/provenance.


In [2]:
def run_cmd(cmd, cwd=None, check=True):
    print('>>>', ' '.join(map(str, cmd)))
    t0 = time.time()
    result = subprocess.run(cmd, cwd=str(cwd) if cwd is not None else None, text=True)
    print(f'Done in {(time.time() - t0) / 60:.1f} min, returncode={result.returncode}')
    if check and result.returncode != 0:
        raise subprocess.CalledProcessError(result.returncode, cmd)
    return result


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
    except Exception as exc:
        print('Torch seed setup skipped:', repr(exc))


if RUN_DEP_SETUP:
    if IS_KAGGLE and CLONE_MONAI_REPO:
        if not MONAI_CLONE_DIR.exists():
            clone_result = run_cmd(['git', 'clone', '--depth', '1', MONAI_REPO_URL, str(MONAI_CLONE_DIR)], check=False)
            if clone_result.returncode != 0:
                print('MONAI clone failed or internet is disabled. Continuing with pip package if available.')
        else:
            print('MONAI repo already exists:', MONAI_CLONE_DIR)

    if INSTALL_MONAI:
        # --no-deps keeps Kaggle Torch/CUDA intact. This avoids the NCCL/Torch mismatch seen in the old U-Mamba attempt.
        pip_result = run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-U', '--no-deps', MONAI_PACKAGE_SPEC, 'nibabel', 'tqdm'], check=False)
        if pip_result.returncode != 0:
            print('MONAI pip install failed. Continuing only if a compatible MONAI package is already available.')

seed_everything(SEED)

import torch
import nibabel as nib
import scipy
import monai

print('Python :', sys.version.replace('\n', ' '))
print('Torch  :', torch.__version__)
print('CUDA   :', torch.version.cuda)
print('MONAI  :', monai.__version__)
print('Nibabel:', nib.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


>>> git clone --depth 1 https://github.com/Project-MONAI/MONAI.git /kaggle/working/MONAI


Cloning into '/kaggle/working/MONAI'...


Done in 0.0 min, returncode=0
>>> /usr/bin/python3 -m pip install -q -U --no-deps monai>=1.5.0 nibabel tqdm
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 6.8 MB/s eta 0:00:00
Done in 0.1 min, returncode=0


2026-06-21 13:00:36.283944: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782046836.464316      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782046836.520194      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782046836.932048      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782046836.932085      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782046836.932088      22 computation_placer.cc:177] computation placer alr

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch  : 2.10.0+cu128
CUDA   : 12.8
MONAI  : 1.5.2
Nibabel: 5.4.2
GPU available: True
GPU name: Tesla T4
GPU memory GB: 14.56


## 1. Locate PHE-SICH and build locked split manifest

The notebook accepts the Kaggle dataset folder automatically. If it cannot find `set/` and `label/`, set `USER_PHE_ROOT` in the first cell.


In [3]:
def case_id_from_path(path):
    name = Path(path).name
    name = re.sub(r'\.nii(\.gz)?$', '', name)
    m = re.search(r'(\d{4})', name)
    return m.group(1) if m else name


def has_phe_layout(root):
    root = Path(root)
    return (root / 'set').is_dir() and (root / 'label').is_dir()


def find_phe_root():
    candidates = []
    if USER_PHE_ROOT is not None:
        candidates.append(Path(USER_PHE_ROOT))

    base_roots = [
        LOCAL_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
        WORK_ROOT / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
    ]
    candidates.extend(base_roots)

    if IS_KAGGLE and KAGGLE_INPUT.exists():
        for dataset_root in sorted([p for p in KAGGLE_INPUT.iterdir() if p.is_dir()]):
            candidates.extend([
                dataset_root / 'PHE-SICH-CT-IDS' / 'SubdatasetA_NIFIT' / 'NIFIT',
                dataset_root / 'SubdatasetA_NIFIT' / 'NIFIT',
                dataset_root / 'NIFIT',
                dataset_root,
            ])

    seen = set()
    for cand in candidates:
        cand = cand.resolve() if cand.exists() else cand
        if str(cand) in seen:
            continue
        seen.add(str(cand))
        if has_phe_layout(cand):
            return cand

    if IS_KAGGLE and KAGGLE_INPUT.exists():
        for cand in KAGGLE_INPUT.rglob('NIFIT'):
            if has_phe_layout(cand):
                return cand

    raise FileNotFoundError('Could not find PHE-SICH NIFIT root with set/ and label/. Set USER_PHE_ROOT manually.')


def build_index(folder):
    files = sorted(list(Path(folder).glob('*.nii')) + list(Path(folder).glob('*.nii.gz')))
    return {case_id_from_path(p): p for p in files}


def split_name(case_id):
    if case_id in TEST_SCAN_IDS:
        return 'test'
    if case_id in VAL_SCAN_IDS:
        return 'val'
    if case_id in TRAIN_SCAN_IDS:
        return 'train'
    raise ValueError(f'Case {case_id} is not in the locked split.')

PHE_ROOT = find_phe_root()
IMAGE_DIR = PHE_ROOT / 'set'
MASK_DIR = PHE_ROOT / 'label'
image_by_id = build_index(IMAGE_DIR)
mask_by_id = build_index(MASK_DIR)

missing_images = sorted(EXPECTED_SCAN_IDS - set(image_by_id))
missing_masks = sorted(EXPECTED_SCAN_IDS - set(mask_by_id))
if missing_images:
    raise FileNotFoundError(f'Missing expected CT images: {missing_images[:20]}')
if missing_masks:
    raise FileNotFoundError(f'Missing expected PHE masks: {missing_masks[:20]}')

records = []
for case_id in sorted(EXPECTED_SCAN_IDS):
    records.append({
        'case_id': case_id,
        'split': split_name(case_id),
        'image': str(image_by_id[case_id]),
        'label': str(mask_by_id[case_id]),
    })

split_df = pd.DataFrame(records)
split_csv = OUTPUT_ROOT / '02_17_swinunetr_locked_split_manifest.csv'
split_df.to_csv(split_csv, index=False)

print('PHE_ROOT:', PHE_ROOT)
print('Images  :', len(image_by_id))
print('Masks   :', len(mask_by_id))
print('Manifest:', split_csv)
print(split_df.groupby('split').size())
display(split_df.head())


PHE_ROOT: /kaggle/input/datasets/onanhv/phe-sich-ct-ids/PHE-SICH-CT-IDS/SubdatasetA_NIFIT/NIFIT
Images  : 120
Masks   : 120
Manifest: /kaggle/working/outputs_02_17_kaggle_swinunetr_phe_only_fair_baseline/02_17_swinunetr_locked_split_manifest.csv
split
test     11
train    99
val      10
dtype: int64


,case_id,split,image,label
0,0001,train,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...
1,0002,test,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...
2,0004,train,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...
3,0005,train,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...
4,0006,train,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...,/kaggle/input/datasets/onanhv/phe-sich-ct-ids/...


## 2. Quick data sanity check

This is not required for training, but it confirms shape/spacing and whether masks are non-empty.


In [4]:
def load_nifti_info(path):
    img = nib.load(str(path))
    arr = np.asanyarray(img.dataobj)
    return {
        'shape': tuple(int(x) for x in arr.shape),
        'spacing': tuple(float(x) for x in img.header.get_zooms()[:3]),
        'min': float(np.nanmin(arr)),
        'max': float(np.nanmax(arr)),
        'nonzero': int(np.count_nonzero(arr)),
    }

sample_rows = []
for row in split_df.groupby('split', sort=False).head(2).itertuples(index=False):
    image_info = load_nifti_info(row.image)
    label_info = load_nifti_info(row.label)
    sample_rows.append({
        'case_id': row.case_id,
        'split': row.split,
        'image_shape': image_info['shape'],
        'spacing': image_info['spacing'],
        'ct_min': image_info['min'],
        'ct_max': image_info['max'],
        'phe_voxels': label_info['nonzero'],
    })

qc_df = pd.DataFrame(sample_rows)
qc_csv = OUTPUT_ROOT / '02_17_swinunetr_data_qc_sample.csv'
qc_df.to_csv(qc_csv, index=False)
print('QC sample:', qc_csv)
display(qc_df)


QC sample: /kaggle/working/outputs_02_17_kaggle_swinunetr_phe_only_fair_baseline/02_17_swinunetr_data_qc_sample.csv


,case_id,split,image_shape,spacing,ct_min,ct_max,phe_voxels
0,0001,train,"(512, 512, 32)","(0.5078125, 0.5078125, 5.0)",-1024.0,1874.0,7962
1,0002,test,"(512, 512, 32)","(0.48828125, 0.48828125, 5.0)",-1024.0,3071.0,1635
2,0004,train,"(512, 512, 32)","(0.4296875, 0.4296875, 5.0)",-1024.0,3071.0,3441
3,0013,val,"(512, 512, 32)","(0.50390625, 0.50390625, 5.0)",-1024.0,1854.0,7467
4,0014,val,"(512, 512, 32)","(0.4296875, 0.4296875, 5.0)",-1024.0,1826.0,5070
5,0029,test,"(512, 512, 32)","(0.4296875, 0.4296875, 5.0)",-1024.0,2201.0,4222


## 3. MONAI datasets and transforms

We train from random 3D patches but validate/test with full-volume sliding-window inference. No extra label source is used.


In [5]:
import torch
from torch.utils.data import DataLoader
from monai.data import CacheDataset, Dataset, list_data_collate
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    ScaleIntensityRanged,
    Lambdad,
    SpatialPadd,
    RandCropByPosNegLabeld,
    RandFlipd,
    RandRotate90d,
    RandScaleIntensityd,
    RandShiftIntensityd,
    EnsureTyped,
)

train_files = split_df[split_df['split'] == 'train'][['image', 'label', 'case_id']].to_dict('records')
val_files = split_df[split_df['split'] == 'val'][['image', 'label', 'case_id']].to_dict('records')
test_files = split_df[split_df['split'] == 'test'][['image', 'label', 'case_id']].to_dict('records')

base_transforms = [
    LoadImaged(keys=['image', 'label'], image_only=True),
    EnsureChannelFirstd(keys=['image', 'label']),
    ScaleIntensityRanged(keys=['image'], a_min=-100.0, a_max=200.0, b_min=0.0, b_max=1.0, clip=True),
    Lambdad(keys=['label'], func=lambda x: (x > 0).astype(np.int64)),
    SpatialPadd(keys=['image', 'label'], spatial_size=ROI_SIZE),
]

train_transforms = Compose(base_transforms + [
    RandCropByPosNegLabeld(
        keys=['image', 'label'],
        label_key='label',
        spatial_size=ROI_SIZE,
        pos=2,
        neg=1,
        num_samples=NUM_SAMPLES_PER_CASE,
        image_key='image',
        image_threshold=0.01,
    ),
    RandFlipd(keys=['image', 'label'], prob=0.50, spatial_axis=0),
    RandFlipd(keys=['image', 'label'], prob=0.50, spatial_axis=1),
    RandRotate90d(keys=['image', 'label'], prob=0.20, max_k=3, spatial_axes=(0, 1)),
    RandScaleIntensityd(keys=['image'], factors=0.10, prob=0.15),
    RandShiftIntensityd(keys=['image'], offsets=0.10, prob=0.15),
    EnsureTyped(keys=['image', 'label'], dtype=torch.float32, track_meta=False),
])

infer_transforms = Compose(base_transforms + [
    EnsureTyped(keys=['image', 'label'], dtype=torch.float32, track_meta=False),
])

DatasetClassTrain = CacheDataset if CACHE_RATE_TRAIN > 0 else Dataset
DatasetClassVal = CacheDataset if CACHE_RATE_VAL_TEST > 0 else Dataset

train_ds = DatasetClassTrain(data=train_files, transform=train_transforms, cache_rate=CACHE_RATE_TRAIN, num_workers=NUM_WORKERS) if DatasetClassTrain is CacheDataset else DatasetClassTrain(data=train_files, transform=train_transforms)
val_ds = DatasetClassVal(data=val_files, transform=infer_transforms, cache_rate=CACHE_RATE_VAL_TEST, num_workers=NUM_WORKERS) if DatasetClassVal is CacheDataset else DatasetClassVal(data=val_files, transform=infer_transforms)
test_ds = DatasetClassVal(data=test_files, transform=infer_transforms, cache_rate=CACHE_RATE_VAL_TEST, num_workers=NUM_WORKERS) if DatasetClassVal is CacheDataset else DatasetClassVal(data=test_files, transform=infer_transforms)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=list_data_collate,
)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print('Train/val/test cases:', len(train_files), len(val_files), len(test_files))
first_batch = next(iter(train_loader))
print('Example train image batch:', tuple(first_batch['image'].shape))
print('Example train label batch:', tuple(first_batch['label'].shape), first_batch['label'].dtype)


Loading dataset: 100%|██████████| 5/5 [00:01<00:00,  3.94it/s]

Train/val/test cases: 99 10 11


Example train image batch: (2, 1, 96, 96, 32)
Example train label batch: (2, 1, 96, 96, 32) torch.float32


## 4. Model and training helpers

The model is MONAI `SwinUNETR`, a 3D Swin Transformer encoder with U-Net style decoder. Feature size and ROI are chosen to fit a Kaggle T4.


In [6]:
import inspect
from torch.cuda.amp import GradScaler, autocast
from monai.networks.nets import SwinUNETR
from monai.losses import DiceCELoss
from monai.inferers import sliding_window_inference
from monai.metrics import DiceMetric
from monai.transforms import AsDiscrete, Activations, Compose as MonaiCompose


def create_model():
    params = inspect.signature(SwinUNETR).parameters
    kwargs = {
        'in_channels': 1,
        'out_channels': 2,
        'feature_size': FEATURE_SIZE,
        'use_checkpoint': USE_GRADIENT_CHECKPOINTING,
    }
    if 'img_size' in params:
        kwargs['img_size'] = ROI_SIZE
    if 'spatial_dims' in params:
        kwargs['spatial_dims'] = 3
    return SwinUNETR(**kwargs)


def checkpoint_payload(epoch, model, optimizer, scheduler, best_val_dice, history):
    return {
        'epoch': int(epoch),
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'best_val_dice': float(best_val_dice),
        'history': history,
        'config': {
            'model_name': MODEL_NAME,
            'roi_size': ROI_SIZE,
            'feature_size': FEATURE_SIZE,
            'max_epochs': MAX_EPOCHS,
            'val_interval': VAL_INTERVAL,
            'batch_size': BATCH_SIZE,
            'num_samples_per_case': NUM_SAMPLES_PER_CASE,
            'learning_rate': LEARNING_RATE,
            'weight_decay': WEIGHT_DECAY,
            'split_sizes': {'train': len(train_files), 'val': len(val_files), 'test': len(test_files)},
            'seed': SEED,
        },
    }


def load_checkpoint_if_available(model, optimizer=None, scheduler=None):
    latest = CHECKPOINT_DIR / 'checkpoint_latest.pt'
    if not (RESUME_TRAINING and latest.exists()):
        return 0, -1.0, []
    try:
        ckpt = torch.load(latest, map_location='cpu', weights_only=False)
    except TypeError:
        ckpt = torch.load(latest, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    if optimizer is not None and ckpt.get('optimizer_state_dict') is not None:
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if scheduler is not None and ckpt.get('scheduler_state_dict') is not None:
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = int(ckpt.get('epoch', -1)) + 1
    best_val_dice = float(ckpt.get('best_val_dice', -1.0))
    history = ckpt.get('history', [])
    print('Resumed from:', latest)
    print('Start epoch:', start_epoch, 'best_val_dice:', best_val_dice)
    return start_epoch, best_val_dice, history


post_pred = MonaiCompose([Activations(softmax=True), AsDiscrete(argmax=True, to_onehot=2)])
post_label = AsDiscrete(to_onehot=2)
dice_metric = DiceMetric(include_background=False, reduction='mean')


def foreground_dice_from_logits(logits, labels):
    pred = torch.argmax(logits, dim=1, keepdim=True) == 1
    gt = labels.long()
    if gt.ndim == 4:
        gt = gt.unsqueeze(1)
    gt = gt == 1
    reduce_dims = tuple(range(1, pred.ndim))
    intersection = (pred & gt).sum(dim=reduce_dims).float()
    denom = pred.sum(dim=reduce_dims).float() + gt.sum(dim=reduce_dims).float()
    return torch.where(denom > 0, 2.0 * intersection / denom, torch.ones_like(denom))


def run_validation(model, loader, device):
    model.eval()
    case_dices = []
    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device).long()
            with autocast(enabled=AMP and device.type == 'cuda'):
                outputs = sliding_window_inference(
                    images,
                    roi_size=ROI_SIZE,
                    sw_batch_size=SW_BATCH_SIZE,
                    predictor=model,
                    overlap=SLIDING_OVERLAP,
                    mode='gaussian',
                )
            case_dices.extend(foreground_dice_from_logits(outputs.detach(), labels).detach().cpu().tolist())
    if not case_dices:
        return 0.0
    return float(np.mean(case_dices))


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if IS_KAGGLE and device.type != 'cuda':
    raise RuntimeError('Kaggle GPU is not available. Enable T4 GPU before training.')

model = create_model().to(device)
loss_function = DiceCELoss(to_onehot_y=True, softmax=True, include_background=False)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
scaler = GradScaler(enabled=AMP and device.type == 'cuda')

num_params = sum(p.numel() for p in model.parameters())
print('Device:', device)
print('Model :', type(model).__name__)
print('Params:', f'{num_params / 1e6:.2f}M')


Device: cuda
Model : SwinUNETR
Params: 15.70M


/tmp/ipykernel_22/365232468.py:117: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=AMP and device.type == 'cuda')


## 5. Train

Default is 250 epochs to match the 02_12-style training budget. If Kaggle runtime is tight, keep the same split and record any reduced epoch budget in the final comparison table.


In [7]:
if RUN_TRAIN:
    start_epoch, best_val_dice, history = load_checkpoint_if_available(model, optimizer, scheduler)
    latest_ckpt = CHECKPOINT_DIR / 'checkpoint_latest.pt'
    best_ckpt = CHECKPOINT_DIR / 'checkpoint_best.pt'

    print('Training from epoch:', start_epoch)
    print('Best val Dice so far:', best_val_dice)
    t0_all = time.time()

    for epoch in range(start_epoch, MAX_EPOCHS):
        model.train()
        epoch_loss = 0.0
        step_count = 0
        t0 = time.time()

        for batch in train_loader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device).long()

            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=AMP and device.type == 'cuda'):
                outputs = model(images)
                loss = loss_function(outputs, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()

            epoch_loss += float(loss.detach().cpu())
            step_count += 1

        scheduler.step()
        mean_loss = epoch_loss / max(step_count, 1)
        record = {
            'epoch': epoch + 1,
            'train_loss': mean_loss,
            'lr': float(optimizer.param_groups[0]['lr']),
            'seconds': round(time.time() - t0, 2),
        }

        should_validate = ((epoch + 1) % VAL_INTERVAL == 0) or (epoch + 1 == MAX_EPOCHS)
        if should_validate:
            val_dice = run_validation(model, val_loader, device)
            record['val_dice'] = val_dice
            print(f"Epoch {epoch + 1:03d}/{MAX_EPOCHS} loss={mean_loss:.4f} val_dice={val_dice:.4f} lr={record['lr']:.2e} time={record['seconds']}s")
            if val_dice > best_val_dice:
                best_val_dice = val_dice
                torch.save(checkpoint_payload(epoch, model, optimizer, scheduler, best_val_dice, history + [record]), best_ckpt)
                print('Saved new best checkpoint:', best_ckpt)
        else:
            print(f"Epoch {epoch + 1:03d}/{MAX_EPOCHS} loss={mean_loss:.4f} lr={record['lr']:.2e} time={record['seconds']}s")

        history.append(record)
        torch.save(checkpoint_payload(epoch, model, optimizer, scheduler, best_val_dice, history), latest_ckpt)
        pd.DataFrame(history).to_csv(METRIC_DIR / '02_17_swinunetr_training_history.csv', index=False)

    print('Training done in min:', round((time.time() - t0_all) / 60, 1))
    print('Best val Dice:', best_val_dice)
else:
    print('Skipped training. Set RUN_TRAIN=True to train.')


Training from epoch: 0
Best val Dice so far: -1.0


/tmp/ipykernel_22/2148265258.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=AMP and device.type == 'cuda'):


Epoch 001/250 loss=1.3165 lr=1.00e-04 time=35.38s
Epoch 002/250 loss=1.2052 lr=1.00e-04 time=31.93s
Epoch 003/250 loss=1.1653 lr=1.00e-04 time=33.93s
Epoch 004/250 loss=1.1384 lr=9.99e-05 time=32.13s


/tmp/ipykernel_22/365232468.py:94: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=AMP and device.type == 'cuda'):
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:226: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  win_data = torch.cat([inputs[win_slice] for win_slice in unravel_slice]).to(sw_device)
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:370: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 thi

Epoch 005/250 loss=1.1111 val_dice=0.0142 lr=9.99e-05 time=32.61s
Saved new best checkpoint: /kaggle/working/outputs_02_17_kaggle_swinunetr_phe_only_fair_baseline/checkpoints/checkpoint_best.pt
Epoch 006/250 loss=1.0881 lr=9.99e-05 time=32.6s
Epoch 007/250 loss=1.0669 lr=9.98e-05 time=32.66s
Epoch 008/250 loss=1.0562 lr=9.97e-05 time=32.25s
Epoch 009/250 loss=1.0437 lr=9.97e-05 time=32.26s
Epoch 010/250 loss=1.0272 val_dice=0.0157 lr=9.96e-05 time=32.54s
Saved new best checkpoint: /kaggle/working/outputs_02_17_kaggle_swinunetr_phe_only_fair_baseline/checkpoints/checkpoint_best.pt
Epoch 011/250 loss=1.0155 lr=9.95e-05 time=32.34s
Epoch 012/250 loss=1.0031 lr=9.94e-05 time=32.69s
Epoch 013/250 loss=0.9933 lr=9.93e-05 time=32.47s
Epoch 014/250 loss=0.9795 lr=9.92e-05 time=32.51s
Epoch 015/250 loss=0.9698 val_dice=0.0258 lr=9.91e-05 time=33.32s
Saved new best checkpoint: /kaggle/working/outputs_02_17_kaggle_swinunetr_phe_only_fair_baseline/checkpoints/checkpoint_best.pt
Epoch 016/250 loss=

## 6. Predict test set and evaluate

Inference uses full-volume sliding-window prediction. Metrics are computed on the locked 11 test cases.


In [8]:
from scipy.ndimage import binary_erosion, generate_binary_structure, distance_transform_edt


def dice_score(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    denom = pred.sum() + gt.sum()
    if denom == 0:
        return 1.0
    return float(2.0 * np.logical_and(pred, gt).sum() / denom)


def binary_metrics(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    tp = int(np.logical_and(pred, gt).sum())
    fp = int(np.logical_and(pred, ~gt).sum())
    fn = int(np.logical_and(~pred, gt).sum())
    tn = int(np.logical_and(~pred, ~gt).sum())
    precision = tp / (tp + fp) if (tp + fp) else (1.0 if gt.sum() == 0 else 0.0)
    recall = tp / (tp + fn) if (tp + fn) else (1.0 if pred.sum() == 0 else 0.0)
    specificity = tn / (tn + fp) if (tn + fp) else 1.0
    return tp, fp, fn, tn, float(precision), float(recall), float(specificity)


def hd95_mm(pred, gt, spacing):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    if not pred.any() and not gt.any():
        return 0.0
    if pred.any() != gt.any():
        return float('inf')
    structure = generate_binary_structure(3, 1)
    pred_surface = np.logical_xor(pred, binary_erosion(pred, structure=structure, border_value=0))
    gt_surface = np.logical_xor(gt, binary_erosion(gt, structure=structure, border_value=0))
    if not pred_surface.any() or not gt_surface.any():
        return float('inf')
    dt_gt = distance_transform_edt(~gt_surface, sampling=spacing)
    dt_pred = distance_transform_edt(~pred_surface, sampling=spacing)
    distances = np.concatenate([dt_gt[pred_surface], dt_pred[gt_surface]])
    return float(np.percentile(distances, 95))


def volume_ml(mask, spacing):
    return float(mask.astype(bool).sum() * np.prod(spacing) / 1000.0)


def load_best_for_inference(model):
    best_ckpt = CHECKPOINT_DIR / 'checkpoint_best.pt'
    latest_ckpt = CHECKPOINT_DIR / 'checkpoint_latest.pt'
    ckpt_path = best_ckpt if best_ckpt.exists() else latest_ckpt
    if not ckpt_path.exists():
        raise FileNotFoundError(f'No checkpoint found in {CHECKPOINT_DIR}. Train first or upload checkpoint.')
    try:
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    except TypeError:
        ckpt = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    print('Loaded checkpoint:', ckpt_path)
    print('Checkpoint best_val_dice:', ckpt.get('best_val_dice'))
    return ckpt_path


if RUN_PREDICT_AND_EVALUATE:
    ckpt_path = load_best_for_inference(model)
    model.to(device)
    model.eval()
    PRED_DIR.mkdir(parents=True, exist_ok=True)

    rows = []
    with torch.no_grad():
        for batch, info in zip(test_loader, test_files):
            case_id = info['case_id']
            images = batch['image'].to(device)
            with autocast(enabled=AMP and device.type == 'cuda'):
                logits = sliding_window_inference(
                    images,
                    roi_size=ROI_SIZE,
                    sw_batch_size=SW_BATCH_SIZE,
                    predictor=model,
                    overlap=SLIDING_OVERLAP,
                    mode='gaussian',
                )
            pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

            gt_nii = nib.load(str(info['label']))
            gt = (np.asanyarray(gt_nii.dataobj) > 0).astype(np.uint8)
            image_nii = nib.load(str(info['image']))
            spacing = tuple(float(x) for x in image_nii.header.get_zooms()[:3])
            original_shape = gt.shape
            pred = pred[:original_shape[0], :original_shape[1], :original_shape[2]]

            pred_path = PRED_DIR / f'{case_id}.nii.gz'
            pred_img = nib.Nifti1Image(pred.astype(np.uint8), affine=gt_nii.affine, header=gt_nii.header)
            pred_img.set_data_dtype(np.uint8)
            nib.save(pred_img, str(pred_path))

            tp, fp, fn, tn, precision, recall, specificity = binary_metrics(pred, gt)
            rows.append({
                'case_id': case_id,
                'checkpoint': str(ckpt_path),
                'prediction_path': str(pred_path),
                'dice': dice_score(pred, gt),
                'precision': precision,
                'sensitivity': recall,
                'recall': recall,
                'specificity': specificity,
                'hd95_mm': hd95_mm(pred, gt, spacing),
                'gt_volume_ml': volume_ml(gt, spacing),
                'pred_volume_ml': volume_ml(pred, spacing),
                'tp': tp,
                'fp': fp,
                'fn': fn,
                'tn': tn,
                'spacing': spacing,
                'shape': original_shape,
            })
            print(f"{case_id}: Dice={rows[-1]['dice']:.4f}, sens={recall:.4f}, pred_ml={rows[-1]['pred_volume_ml']:.2f}, gt_ml={rows[-1]['gt_volume_ml']:.2f}")

    metrics_df = pd.DataFrame(rows)
    metrics_csv = METRIC_DIR / '02_17_swinunetr_test_metrics.csv'
    metrics_df.to_csv(metrics_csv, index=False)

    finite_hd95 = metrics_df['hd95_mm'].replace([np.inf, -np.inf], np.nan)
    summary = {
        'model': MODEL_NAME,
        'checkpoint': str(ckpt_path),
        'n_test': int(len(metrics_df)),
        'dice_mean': float(metrics_df['dice'].mean()),
        'dice_median': float(metrics_df['dice'].median()),
        'sensitivity_mean': float(metrics_df['sensitivity'].mean()),
        'precision_mean': float(metrics_df['precision'].mean()),
        'specificity_mean': float(metrics_df['specificity'].mean()),
        'hd95_mm_mean_finite': float(finite_hd95.mean()) if finite_hd95.notna().any() else None,
        'gt_volume_ml_mean': float(metrics_df['gt_volume_ml'].mean()),
        'pred_volume_ml_mean': float(metrics_df['pred_volume_ml'].mean()),
        'fairness': {
            'dataset': 'PHE-SICH CT only',
            'target': 'manual PHE binary mask',
            'train_cases': len(train_files),
            'val_cases': len(val_files),
            'test_cases': len(test_files),
            'no_ich_teacher': True,
            'no_pseudo_ich': True,
            'no_prior_channel': True,
        },
    }
    summary_json = METRIC_DIR / '02_17_swinunetr_test_summary.json'
    summary_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')

    print('Metrics CSV :', metrics_csv)
    print('Summary JSON:', summary_json)
    print(json.dumps(summary, indent=2))
    display(metrics_df)
else:
    print('Skipped prediction/evaluation. Set RUN_PREDICT_AND_EVALUATE=True after training.')


Loaded checkpoint: /kaggle/working/outputs_02_17_kaggle_swinunetr_phe_only_fair_baseline/checkpoints/checkpoint_best.pt
Checkpoint best_val_dice: 0.19829939156770707


/tmp/ipykernel_22/3986440809.py:75: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=AMP and device.type == 'cuda'):


0002: Dice=0.0436, sens=0.0422, pred_ml=1.82, gt_ml=1.95
0029: Dice=0.2021, sens=0.3020, pred_ml=7.75, gt_ml=3.90
0033: Dice=0.0925, sens=0.0662, pred_ml=2.85, gt_ml=6.61
0051: Dice=0.0008, sens=0.0032, pred_ml=2.65, gt_ml=0.37
0061: Dice=0.0000, sens=0.0000, pred_ml=3.95, gt_ml=0.49
0084: Dice=0.4220, sens=0.5215, pred_ml=4.05, gt_ml=2.75
0095: Dice=0.1677, sens=0.1725, pred_ml=5.99, gt_ml=5.66
0096: Dice=0.1801, sens=0.1250, pred_ml=2.97, gt_ml=7.65
0113: Dice=0.0000, sens=0.0000, pred_ml=1.56, gt_ml=0.29
0116: Dice=0.1707, sens=0.4599, pred_ml=3.46, gt_ml=0.79
0167: Dice=0.0000, sens=0.0000, pred_ml=1.06, gt_ml=1.09
Metrics CSV : /kaggle/working/outputs_02_17_kaggle_swinunetr_phe_only_fair_baseline/metrics/02_17_swinunetr_test_metrics.csv
Summary JSON: /kaggle/working/outputs_02_17_kaggle_swinunetr_phe_only_fair_baseline/metrics/02_17_swinunetr_test_summary.json
{
  "model": "monai_swinunetr_3d_phe_only",
  "checkpoint": "/kaggle/working/outputs_02_17_kaggle_swinunetr_phe_only_fair_

,case_id,checkpoint,prediction_path,dice,precision,sensitivity,recall,specificity,hd95_mm,gt_volume_ml,pred_volume_ml,tp,fp,fn,tn,spacing,shape
0,0002,/kaggle/working/outputs_02_17_kaggle_swinunetr...,/kaggle/working/outputs_02_17_kaggle_swinunetr...,0.043616,0.045128,0.042202,0.042202,0.999826,58.238526,1.949072,1.822710,69,1460,1566,8385513,"(0.48828125, 0.48828125, 5.0)","(512, 512, 32)"
1,0029,/kaggle/working/outputs_02_17_kaggle_swinunetr...,/kaggle/working/outputs_02_17_kaggle_swinunetr...,0.202108,0.151876,0.301990,0.301990,0.999151,34.526672,3.897568,7.749901,1275,7120,2947,8377266,"(0.4296875, 0.4296875, 5.0)","(512, 512, 32)"
2,0033,/kaggle/working/outputs_02_17_kaggle_swinunetr...,/kaggle/working/outputs_02_17_kaggle_swinunetr...,0.092502,0.153428,0.066210,0.066210,0.999758,55.694021,6.607771,2.851486,367,2025,5176,8381040,"(0.48828125, 0.48828125, 5.0)","(512, 512, 32)"
3,0051,/kaggle/working/outputs_02_17_kaggle_swinunetr...,/kaggle/working/outputs_02_17_kaggle_swinunetr...,0.000789,0.000449,0.003247,0.003247,0.999660,33.958262,0.367165,2.654791,1,2226,307,6551066,"(0.48828125, 0.48828125, 5.0)","(512, 512, 25)"
4,0061,/kaggle/working/outputs_02_17_kaggle_swinunetr...,/kaggle/working/outputs_02_17_kaggle_swinunetr...,0.000000,0.000000,0.000000,0.000000,0.999494,73.754524,0.485182,3.954172,0,3317,407,6549876,"(0.48828125, 0.48828125, 5.0)","(512, 512, 25)"
5,0084,/kaggle/working/outputs_02_17_kaggle_swinunetr...,/kaggle/working/outputs_02_17_kaggle_swinunetr...,0.422021,0.354437,0.521452,0.521452,0.999691,63.891055,2.751509,4.048058,1422,2590,1305,8383291,"(0.44921875, 0.44921875, 5.0)","(512, 512, 32)"
6,0095,/kaggle/working/outputs_02_17_kaggle_swinunetr...,/kaggle/working/outputs_02_17_kaggle_swinunetr...,0.167671,0.163063,0.172547,0.172547,0.999617,57.752006,5.660169,5.989358,626,3213,3002,8381767,"(0.55859375, 0.55859375, 5.0)","(512, 512, 32)"
7,0096,/kaggle/working/outputs_02_17_kaggle_swinunetr...,/kaggle/working/outputs_02_17_kaggle_swinunetr...,0.180124,0.322218,0.125000,0.125000,0.999785,29.331441,7.648468,2.967119,802,1687,5614,7856217,"(0.48828125, 0.48828125, 5.0)","(512, 512, 30)"
8,0113,/kaggle/working/outputs_02_17_kaggle_swinunetr...,/kaggle/working/outputs_02_17_kaggle_swinunetr...,0.000000,0.000000,0.000000,0.000000,0.999801,53.984394,0.286102,1.555681,0,1305,240,6552055,"(0.48828125, 0.48828125, 5.0)","(512, 512, 25)"
9,0116,/kaggle/working/outputs_02_17_kaggle_swinunetr...,/kaggle/working/outputs_02_17_kaggle_swinunetr...,0.170691,0.104791,0.459909,0.459909,0.999690,41.285484,0.787973,3.458261,304,2597,357,8385350,"(0.48828125, 0.48828125, 5.0)","(512, 512, 32)"


## 7. Experiment registry

This small registry row makes it easier to compare against 02_12, 16b, 16c, 18, 13+14, and 13b+14b in the report.


In [9]:
registry = {
    'experiment': '02_17',
    'notebook': '02_17_kaggle_swinunetr_phe_only_fair_baseline.ipynb',
    'model': MODEL_NAME,
    'architecture': '3D SwinUNETR / 3D Swin Transformer U-Net family',
    'framework': 'MONAI',
    'run_target': 'Kaggle T4',
    'dataset': 'PHE-SICH CT-IDS SubdatasetA_NIFIT/NIFIT',
    'input_channels': 'CT only',
    'label': 'manual PHE binary mask',
    'train_cases': len(train_files),
    'val_cases': len(val_files),
    'test_cases': len(test_files),
    'split_source': 'locked 02_12 split ids',
    'max_epochs': MAX_EPOCHS,
    'roi_size': str(ROI_SIZE),
    'feature_size': FEATURE_SIZE,
    'amp': AMP,
    'no_ich_teacher': True,
    'no_pseudo_ich': True,
    'no_prior_channel': True,
    'output_root': str(OUTPUT_ROOT),
}

summary_path = METRIC_DIR / '02_17_swinunetr_test_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    registry.update({
        'test_dice_mean': summary.get('dice_mean'),
        'test_dice_median': summary.get('dice_median'),
        'test_sensitivity_mean': summary.get('sensitivity_mean'),
        'test_precision_mean': summary.get('precision_mean'),
        'test_hd95_mm_mean_finite': summary.get('hd95_mm_mean_finite'),
    })

registry_csv = OUTPUT_ROOT / '02_17_swinunetr_experiment_registry.csv'
pd.DataFrame([registry]).to_csv(registry_csv, index=False)
print('Registry:', registry_csv)
display(pd.DataFrame([registry]))


Registry: /kaggle/working/outputs_02_17_kaggle_swinunetr_phe_only_fair_baseline/02_17_swinunetr_experiment_registry.csv


,experiment,notebook,model,architecture,framework,run_target,dataset,input_channels,label,train_cases,...,amp,no_ich_teacher,no_pseudo_ich,no_prior_channel,output_root,test_dice_mean,test_dice_median,test_sensitivity_mean,test_precision_mean,test_hd95_mm_mean_finite
0,02_17,02_17_kaggle_swinunetr_phe_only_fair_baseline....,monai_swinunetr_3d_phe_only,3D SwinUNETR / 3D Swin Transformer U-Net family,MONAI,Kaggle T4,PHE-SICH CT-IDS SubdatasetA_NIFIT/NIFIT,CT only,manual PHE binary mask,99,...,True,True,True,True,/kaggle/working/outputs_02_17_kaggle_swinunetr...,0.11632,0.092502,0.153869,0.117763,53.420477


## Notes for comparison

Use this notebook as the architecture-control baseline:

- Compare to `02_12`: same PHE-only task and split, different architecture/training framework.
- Compare to `16b/16c/18`: keep focus on PHE Dice/sensitivity/precision on the same locked test set.
- Do not compare directly to ICH-prior experiments as if labels are identical unless the report states the changed input information clearly.
- If Kaggle time forces fewer than 250 epochs, record the actual epoch count from `02_17_swinunetr_training_history.csv` before adding the result to `baocao.md`.
